<a href="https://colab.research.google.com/github/Gyuseo-stack/Statistics_DGS/blob/main/Food_delivery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
gauravmalik26_food_delivery_dataset_path = kagglehub.dataset_download('gauravmalik26/food-delivery-dataset')

print('Data source import complete.')


Using Colab cache for faster access to the 'food-delivery-dataset' dataset.
Data source import complete.


* This notebook is demonstration of creating new feature on the basis of longitude and latitude.
* Feature engineering on the basis of aggregation.
* Target Encoding
* Feature Selection
* Stacking

## IMPORTING

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
gauravmalik26_food_delivery_dataset_path = kagglehub.dataset_download('gauravmalik26/food-delivery-dataset')

print('Data source import complete.')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.impute import SimpleImputer,MissingIndicator, KNNImputer
from sklearn.model_selection import train_test_split, cross_validate,GridSearchCV,RepeatedStratifiedKFold,cross_val_score,cross_val_predict
from sklearn.preprocessing import PowerTransformer,OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import make_column_transformer,make_column_selector,ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, GradientBoostingRegressor,RandomForestClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score
from sklearn.impute import  KNNImputer
from sklearn.svm import SVC
from lightgbm import LGBMRegressor, LGBMClassifier
import xgboost as xgb

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
#from kuma_utils.preprocessing.imputer import LGBMImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
import holidays
import gc
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import SplineTransformer
from sklearn.preprocessing import PolynomialFeatures
from geopy.distance import geodesic

import gc

!pip install category_encoders
!pip install feature_engine
!pip install catboost
from category_encoders import OrdinalEncoder


import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.preprocessing import KBinsDiscretizer,LabelEncoder
from feature_engine.encoding import OrdinalEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropDuplicateFeatures, DropConstantFeatures
from feature_engine.selection import (
    RecursiveFeatureElimination,
    RecursiveFeatureAddition,
    DropConstantFeatures,
    DropDuplicateFeatures,
    SmartCorrelatedSelection,
    DropCorrelatedFeatures
)
from feature_engine.creation import MathFeatures,RelativeFeatures

# to select the features
from sklearn.feature_selection import SelectKBest, SelectPercentile
from feature_engine.encoding import CountFrequencyEncoder
from feature_engine.encoding import OneHotEncoder
from feature_engine.encoding import MeanEncoder
from sklearn import metrics
from feature_engine.encoding import CountFrequencyEncoder
from sklearn.cluster import KMeans
from category_encoders import MEstimateEncoder,OrdinalEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import KBinsDiscretizer

from catboost import CatBoostRegressor # Added CatBoostRegressor import
from sklearn.ensemble import StackingRegressor

Using Colab cache for faster access to the 'food-delivery-dataset' dataset.
Data source import complete.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.2 MB/s eta 0:00:00


In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor ,AdaBoostRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, BayesianRidge, ElasticNet, LassoLars, PassiveAggressiveRegressor
from sklearn.svm import SVR, NuSVR
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor, metrics, Pool, cv
pd.set_option('display.max_columns', None)

## DATA LOADING

In [ ]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

## DATA CLEANING

In [ ]:
#test=test.replace(" ","")
test=test.replace('NaN', float(np.nan), regex=True)

#train=train.replace(" ","")
train=train.replace('NaN', float(np.nan), regex=True)

In [ ]:
train['Weatherconditions']=train['Weatherconditions'].str.split(" ", expand=True)[1]
test['Weatherconditions']=test['Weatherconditions'].str.split(" ", expand=True)[1]

train['Time_taken(min)']=train['Time_taken(min)'].str.split(" ", expand=True)[1]

In [ ]:
num_cols = ['Delivery_person_Age','Delivery_person_Ratings','Restaurant_latitude','Restaurant_longitude',
            'Delivery_location_latitude','Delivery_location_longitude','Vehicle_condition',
            'multiple_deliveries','Time_taken(min)']
for col in num_cols:
    train[col]=train[col].astype('float64')

for col in num_cols[:-1]:
    test[col]=test[col].astype('float64')

train['Order_Date']=pd.to_datetime(train['Order_Date'],format="%d-%m-%Y")
test['Order_Date']=pd.to_datetime(test['Order_Date'],format="%d-%m-%Y")

모델 학습을 위해 문자형 숫자를 실수형(float64) 로 변환

test에는 Time_taken(min)이 없기 때문에 제외 test는 -1인 이유

"11-02-2024" 형태의 날짜를 datetime 형태로 변환

In [ ]:
train['Time_Orderd']=pd.to_timedelta(train['Time_Orderd'])
train['Time_Order_picked']=pd.to_timedelta(train['Time_Order_picked'])

train['Time_Order_picked_formatted']=np.where(train['Time_Order_picked'] < train['Time_Orderd'], train['Order_Date'] + pd.DateOffset(1)+train['Time_Order_picked'], train['Order_Date']+train['Time_Order_picked'])
train['Time_Ordered_formatted'] = train['Order_Date']+ train['Time_Orderd']
train['order_prepare_time_diff_mins']=((train['Time_Order_picked_formatted']- train['Time_Ordered_formatted']).dt.total_seconds())/60

"12:45:00" 이런 문자열을 시간 간격(timedelta) 형태로 변환

픽업 시간이 주문시간보다 작은 경우 하루 추가 (자정 넘어간 케이스 처리)
예:
주문 23:50, 픽업 00:10 → 실제는 다음날 00:10

이 오류를 해결하기 위해 다음날 날짜를 더해줌

픽업까지 걸린 시간(분) 계산하여 파생변수 생성

In [ ]:
test['Time_Orderd']=pd.to_timedelta(test['Time_Orderd'])
test['Time_Order_picked']=pd.to_timedelta(test['Time_Order_picked'])

test['Time_Order_picked_formatted']=np.where(test['Time_Order_picked'] < test['Time_Orderd'], test['Order_Date'] + pd.DateOffset(1)+test['Time_Order_picked'], test['Order_Date']+test['Time_Order_picked'])
test['Time_Ordered_formatted'] = test['Order_Date']+ test['Time_Orderd']
test['order_prepare_time_diff_mins']=((test['Time_Order_picked_formatted']- test['Time_Ordered_formatted']).dt.total_seconds())/60

In [ ]:
cols=['Restaurant_latitude','Restaurant_longitude','Delivery_location_latitude','Delivery_location_longitude']
for col in cols:
    train[col]= abs(train[col])
for col in cols:
    test[col]= abs(test[col])

위도 · 경도는 절대값 처리

## Creating new features on the basis of latitude and longitude

In [ ]:
train['distance_diff_KM']=np.zeros(len(train))

# Identify coordinate columns
coord_cols = ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']

# Temporarily fill NaN values in coordinate columns with their mean for distance calculation
# This prevents the ValueError from geodesic.
train_coords_filled = train[coord_cols].fillna(train[coord_cols].mean())

restaurant_cordinates_train = train_coords_filled[['Restaurant_latitude','Restaurant_longitude']].to_numpy()
delivery_location_cordinates_train = train_coords_filled[['Delivery_location_latitude','Delivery_location_longitude']].to_numpy()

for i in range(len(train)):
    # geodesic returns a Distance object, access the .km attribute to get the numerical value
    train['distance_diff_KM'].loc[i] = geodesic(restaurant_cordinates_train[i], delivery_location_cordinates_train[i]).km

- 식당 위치와 고객 위치(Delivery location) 간의 실제 이동거리(km) 를 계산해 새로운 파생변수를 생성

train['distance_diff_KM']=np.zeros(len(train))
-> 새로운 컬럼(distance_diff_KM)을 미리 만들고 0으로 초기화

- 좌표 컬럼 NaN 값 처리


(위경도에 NaN이 있으면 geodesic 함수가 에러를 발생시키기 때문에
해당 컬럼들의 NaN을 평균값으로 채워서 거리 계산 가능하게 만든다.)

- 위경도 배열 생성
식당 좌표 (lat, lon), 배달 목적지 좌표 (lat, lon)
이 두 가지 배열로 나누어 저장.

- geodesic 거리 계산(두 지점의 위도/경도를 입력받아 실제 지구 표면을 따라간 최단 거리(측지 거리)를 계산하는 함수)

geopy의 geodesic(대권거리) 함수 사용

두 위경도 간 실제 지구 곡률 기반 거리(km)를 계산

각 행마다 반복하며 distance_diff_KM에 저장

In [ ]:
test['distance_diff_KM']=np.zeros(len(test))

# Identify coordinate columns
coord_cols_test = ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']

# Temporarily fill NaN values in coordinate columns with their mean for distance calculation
# This prevents the ValueError from geodesic.
test_coords_filled = test[coord_cols_test].fillna(test[coord_cols_test].mean())

restaurant_cordinates_test = test_coords_filled[['Restaurant_latitude','Restaurant_longitude']].to_numpy()
delivery_location_cordinates_test = test_coords_filled[['Delivery_location_latitude','Delivery_location_longitude']].to_numpy()

for i in range(len(test)):
    # geodesic returns a Distance object, access the .km attribute to get the numerical value
    test['distance_diff_KM'].loc[i]=geodesic(restaurant_cordinates_test[i],delivery_location_cordinates_test[i]).km

test도 위와 똑같이

In [ ]:

y=train["Time_taken(min)"]
train = train.drop(columns=["Time_taken(min)",'ID'],axis=1)
Id= test['ID'].str.strip()
test = test.drop(columns=["ID"],axis=1)


- train 데이터에서 예측해야 할 값인 배달 소요 시간(min) 을 y로 분리
- 학습 시 불필요한 ID 컬럼 제거, 타깃은 y 변수로 따로 관리하므로 train에서 제거
- test에는 타깃값이 없으므로 ID만 제거,이후 제출 파일 생성시 다시 Id를 사용할 수 있도록 저장

(test에는 label이 없으니까 ID를 모델 입력에서 제거하지만, 최종 제출 파일을 만들 때는 예측값을 어떤 사람에게 대응하는지 알아야 하므로 test ID는 따로 저장)

In [ ]:
data=pd.concat([train,test]).reset_index(drop=True)

In [ ]:
data.head()

,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,...,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_Order_picked_formatted,Time_Ordered_formatted,order_prepare_time_diff_mins,distance_diff_KM
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,0 days 11:30:00,0 days 11:45:00,...,2.0,Snack,motorcycle,0.0,No,Urban,2022-03-19 11:45:00,2022-03-19 11:30:00,15.0,3.020737
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,0 days 19:45:00,0 days 19:50:00,...,2.0,Snack,scooter,1.0,No,Metropolitian,2022-03-25 19:50:00,2022-03-25 19:45:00,5.0,20.143737
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,0 days 08:30:00,0 days 08:45:00,...,0.0,Drinks,motorcycle,1.0,No,Urban,2022-03-19 08:45:00,2022-03-19 08:30:00,15.0,1.549693
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,0 days 18:00:00,0 days 18:10:00,...,0.0,Buffet,motorcycle,1.0,No,Metropolitian,2022-04-05 18:10:00,2022-04-05 18:00:00,10.0,7.774497
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,0 days 13:30:00,0 days 13:45:00,...,1.0,Snack,scooter,1.0,No,Metropolitian,2022-03-26 13:45:00,2022-03-26 13:30:00,15.0,6.197898


In [ ]:
na_cols=[]
for col in data.columns:
    if data[col].isna().sum()>0:
        na_cols.append(col)

data["n_missing"]= data[na_cols].isna().sum(axis=1)
data_missing_tag_df = data[na_cols].isna().astype(np.int8)
data_missing_tag_df.columns = [f"{c}_missing" for c in data_missing_tag_df.columns]

data=pd.concat([data, data_missing_tag_df], axis=1)

각 행(row)이 얼마나 많은 결측값을 가지고 있는지 세고,
각 컬럼마다 결측 여부를 표시하는 파생변수를 생성

ML 모델에게 “이 데이터 불완전함 정도”를 알려주는 중요한 신호이다.
✔ (1) 각 행의 결측값 개수(n_missing) 파생변수 생성

→ 데이터 품질 신호 제공

✔ (2) 각 컬럼별 결측 여부(0/1) 파생 컬럼 생성

→ 모델이 결측 패턴을 학습하도록 도와줌

## Creating time based features

In [ ]:
# Before creating time based features, handle NaT values in 'Order_Date'
# Fill NaT with the most frequent date to avoid IntCastingNaNError
if data['Order_Date'].isnull().any():
    mode_date = data['Order_Date'].mode()[0]
    data['Order_Date'] = data['Order_Date'].fillna(mode_date)

data["day"] = data.Order_Date.dt.day
data["week"] = data.Order_Date.dt.isocalendar().week
data["month"] = data.Order_Date.dt.month
data["quarter"] = data.Order_Date.dt.quarter
data["year"] = data.Order_Date.dt.year
data["hour"] = data.Time_Orderd.dt.components['hours']
data["dayofyear"] = data.Order_Date.dt.dayofyear
data['day_of_week'] = data.Order_Date.dt.day_of_week.astype(int)
data["is_month_start"] = data.Order_Date.dt.is_month_start.astype(int)
data["is_month_end"] = data.Order_Date.dt.is_month_end.astype(int)
data["is_quarter_start"] = data.Order_Date.dt.is_quarter_start.astype(int)
data["is_quarter_end"] = data.Order_Date.dt.is_quarter_end.astype(int)
data["is_year_start"] = data.Order_Date.dt.is_year_start.astype(int)
data["is_year_end"] = data.Order_Date.dt.is_year_end.astype(int)
data["is_leap_year"] = data.Order_Date.dt.is_leap_year.astype(int)
data["days_in_month"] = data.Order_Date.dt.days_in_month
data['is_weekend'] = np.where(data['day_of_week'].isin([5,6]),1,0)

Order_Date에 NaT(결측 날짜)가 있으면 가장 많이 등장한 날짜(mode) 로 대체

날짜 관련 파생변수 생성

day, week, month, quarter, year

dayofyear: 1~365

day_of_week: 월=0 ~ 일=6

월 시작/끝 여부 (is_month_start, is_month_end)

분기 시작/끝 여부

연도 시작/끝 여부

윤년 여부

그달의 총 날짜 수

주말 여부 (is_weekend)

주문 시간(Time_Orderd)에서 시간(hour) 만 추출

In [ ]:
data['distance_diff_KM']=data['distance_diff_KM'].astype("str").str.extract('(\d+)')
data['distance_diff_KM']=data['distance_diff_KM'].astype("int64")

geodesic() 계산 후 비정상적인 문자열이 생길 수 있어 숫자만 추출

In [ ]:
data=data.drop(columns=['Order_Date','Time_Orderd','Time_Order_picked'],axis=1)
data=data.drop(columns=['Time_Order_picked_formatted','Time_Ordered_formatted'])

날짜/시간 원본 컬럼 삭제 - 이미 파생변수를 만들었기 때문에 원본은 필요 없음

In [ ]:
data['week']=data['week'].astype("int64")

week → 정수형으로

모든 uint8 컬럼 → int64 로 통일

In [ ]:
for col in data.columns:
    if data[col].dtype == 'uint8':
        data[col]=data[col].astype("int64")

In [ ]:
data['city_code']=data['Delivery_person_ID'].str.split("RES", expand=True)[0]

RES 기준으로 앞부분만 가져옴

In [ ]:
data=data.drop(columns=["Delivery_person_ID" ],  axis=1)

## Removing constant and duplicate features

In [ ]:
pipe = Pipeline([
   ('constant', DropConstantFeatures(tol=0.998,missing_values='ignore')),
   ('duplicated', DropDuplicateFeatures(missing_values='ignore')),
])
data=pipe.fit_transform(data)


## More Feature Engineering

In [ ]:
num_ft = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
mf=MathFeatures(['Delivery_person_Age','Delivery_person_Ratings'],missing_values='ignore',func= ["std",'mean'])
data=mf.fit_transform(data)

Delivery_person_Age와 Delivery_person_Ratings 두 변수에 대해
표준편차(std), 평균(mean) 기반의 자동 파생변수 생성

In [ ]:
feat1=data.groupby("Delivery_person_Age")["Delivery_person_Ratings"].transform("mean")
data['feat2']=data['Delivery_person_Ratings'] - feat1

feat3=data.groupby("Delivery_person_Age")["Delivery_person_Ratings"].transform("std")
data['feat4']=data['Delivery_person_Ratings'] - feat3

data['feat9']=data.groupby(["Weatherconditions",'Road_traffic_density'])["Delivery_person_Age"].transform("mean")
#data['feat10']=data['Delivery_person_Age'] - data['feat9']

feat11=data.groupby(["Weatherconditions",'Road_traffic_density'])["Delivery_person_Age"].transform("std")
data['feat12']=data['Delivery_person_Age'] - feat11

data['feat13']=data.groupby(["Weatherconditions",'Road_traffic_density','City','city_code'])["distance_diff_KM"].transform("std")
data['feat14']=data['Delivery_person_Age'] - data['feat13']



수치형 변수를 더 풍부한 형태로 확장하는 Feature Engineering 단계

같은 나이대의 배달원 평균과 비교해 개별 배달원의 평점이 높은지/낮은지 측정

평균 뿐만 아니라 표준편차 기준으로도 편차를 계산
→ 이상치 성향이나 변동성 반영

좋은 날씨 vs 나쁜 날씨, 교통상황에 따라
배달원 연령대가 달라지는 패턴을 학습하게 함.

그룹 평균/표준편차와의 차이를 통해 "상대적·정규화된 특성"을 만들어 모델의 예측력을 크게 향상


## Target encoding

In [ ]:
encoder = MEstimateEncoder(m=1)

# Get indices where y is not NaN
valid_y_indices = y.dropna().index

# Filter y to remove NaN values
y_cleaned = y.loc[valid_y_indices]

# Split data back into original train and test parts, using the length of original train.
# 'data' still contains the original train features first, then test features.
original_train_features = data.iloc[:len(y)]
original_test_features = data.iloc[len(y):]

# Filter the original_train_features using the valid_y_indices
train_for_encoding = original_train_features.loc[valid_y_indices]

# Fit and transform the encoder
train = encoder.fit_transform(train_for_encoding, y_cleaned)
test = encoder.transform(original_test_features)

# Update y to be the cleaned version for consistency in subsequent steps
y = y_cleaned

타깃값(Time_taken)을 기반으로 범주형 카테고리를 숫자로 변환

단순 평균 인코딩보다 overfitting 방지력이 더 좋음

Weatherconditions, City 같은 범주형 변수를 숫자형 예측친화적 변수로 변환

범주별 평균 배달시간(Time_taken)을 기반으로 수치화
→ 모델 성능이 크게 증가

In [ ]:
xgb_model=xgb.XGBRegressor(n_estimators=25,max_depth=7)
lgb = LGBMRegressor(num_leaves=85,max_depth=8,learning_rate=0.1)
cat= CatBoostRegressor(n_estimators=700,max_depth=8,learning_rate=0.05)

| 모델       | 설명                                     |
| -------- | -------------------------------------- |
| XGBoost  | 대회에서 가장 많이 쓰는 boosting 모델. 과적합에 강함     |
| LightGBM | 속도 빠르고 대용량에 강함                         |
| CatBoost | One-hot, target encoding 없이도 범주형 처리 강력 |


## Feature Selection

In [ ]:
# from mlxtend.feature_selection import SequentialFeatureSelector as SFS
# sfs = SFS(LGBMRegressor(num_leaves=85,max_depth=8,learning_rate=0.1),
#            k_features=20,
#            forward=True,
#            floating=False,
#            verbose=2,
#            scoring='r2',
#            cv=2)

# sfs = sfs.fit(train ,y)
# selected_feat_= list(sfs.k_feature_names_)

# selected_feat_

In [ ]:
selected_feat_=['Delivery_person_Age',
 'Delivery_person_Ratings',
 'Weatherconditions',
 'Road_traffic_density',
 'Vehicle_condition',
 'multiple_deliveries',
 'Festival',
 'City',
 'distance_diff_KM',
 'n_missing',
 'Weatherconditions_missing',
 'multiple_deliveries_missing',
 'Festival_missing',
 'City_missing',
 'is_month_end',
 'is_quarter_start',
 'is_weekend',
 'feat2',
 'feat9',
 'feat12']

SFS가 뽑아 준 가장 중요한 20개의 feature 목록
— 모델 성능(R² 기준)을 가장 높여주는 특징들


SFS - 가장 널리 쓰이는 Feature Selection 기법 중 하나이고, 머신러닝 모델에서 성능이 잘 나오는 중요한 피처만 자동으로 골라주는 알고리즘

피처를 하나씩 추가하거나 제거하면서, 모델 성능이 가장 좋아지도록 최적의 피처 조합을 찾아주는 알고리즘

In [ ]:
imp_feat=selected_feat_[:12]

NameError: name 'selected_feat_' is not defined

20개 중에서 앞의 12개만 골라서 최종 모델에 사용할 핵심 피처 집합 구성

## Stacking

In [ ]:

#best_1

estimators = [
     ('xgb_model', xgb_model),
     ('lgb', lgb),('cat',cat)]
# best
reg = StackingRegressor(
estimators=estimators,
 final_estimator=xgb_model)

xgb_model.fit(train[imp_feat],y)
reg.fit(train,y)
lgb.fit(train[imp_feat],y)
cat.fit(train[imp_feat],y)

reg_pred=reg.predict(test)
lgb_pred=lgb.predict(test[imp_feat])
cat_pred=cat.predict(test[imp_feat])
xgb_pred=xgb_model.predict(test[imp_feat])

ensemble=reg_pred*0.2 + lgb_pred*0.45 + cat_pred*0.35  # Using these ratios after lots of experimenting




스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
136:	learn: 3.7733125	total: 3.22s	remaining: 13.2s
137:	learn: 3.7725223	total: 3.24s	remaining: 13.2s
138:	learn: 3.7714184	total: 3.26s	remaining: 13.2s
139:	learn: 3.7706801	total: 3.29s	remaining: 13.2s
140:	learn: 3.7700094	total: 3.31s	remaining: 13.1s
141:	learn: 3.7690738	total: 3.34s	remaining: 13.1s
142:	learn: 3.7673538	total: 3.36s	remaining: 13.1s
143:	learn: 3.7645006	total: 3.38s	remaining: 13.1s
144:	learn: 3.7642132	total: 3.4s	remaining: 13s
145:	learn: 3.7628889	total: 3.43s	remaining: 13s
146:	learn: 3.7619867	total: 3.45s	remaining: 13s
147:	learn: 3.7613846	total: 3.47s	remaining: 12.9s
148:	learn: 3.7606627	total: 3.5s	remaining: 12.9s
149:	learn: 3.7600233	total: 3.52s	remaining: 12.9s
150:	learn: 3.7592380	total: 3.54s	remaining: 12.9s
151:	learn: 3.7584308	total: 3.56s	remaining: 12.8s
152:	learn: 3.7576244	total: 3.58s	remaining: 12.8s
153:	learn: 3.7568678	total: 3.63s	remaining: 12.9s
154:	learn: 3.7562789	total: 3.67s	r

이 마지막 코드는 여러 모델을 섞어서(앙상블) 최종 예측값을 만드는 과정

기본(base) 모델: XGBoost, LightGBM, CatBoost → 서로 다른 3개의 부스팅 모델

final_estimator: XGBoost → base 모델들의 예측값을 다시 입력받아 최종 예측을 수행하는 메타 모델


각 모델을 훈련 데이터로 학습시키는 과정

train[imp_feat]: SFS로 선정된 중요한 feature 12개만 사용

train: 전체 feature 사용 (Stacking 은 전체 feature 사용)

reg_pred: Stacking model 의 예측 결과

lgb_pred: LightGBM 단일 모델 예측

cat_pred: CatBoost 단일 모델 예측

xgb_pred: XGBoost 단일 모델 예측



## CONCLUSION

**Finally I encourage my fellow Kagglers to use this data in order to improve their ML skills such as**
* create features on the basis of location, use different types of aggreation techinique.
* use different techinque to fill na values
* try to use different transformations
* try different feature selection techniques
* try different encoding techinqiue such as ordinal encoding etc.
* try different alogirithims